# Father and Daughter — Colab T4 worker

Chay **~2 gio moi tuan** cho mot batch 10 video.

**Quy tac:**
1. Runtime > Change runtime type > **T4 GPU**
2. Chay lan luot tu tren xuong
3. Bi ngat giua chung thi **chay lai tu dau notebook** — no tu bo qua phan da xong
4. Dung dong tab khi dang chay. Dung chay qua 1 batch moi phien.


In [ ]:
# 1. Kiem tra GPU. Phai la T4. Neu khong co GPU thi DUNG LAI.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Chua bat GPU: Runtime > Change runtime type > T4'
print('bf16 ho tro:', torch.cuda.is_bf16_supported(), '(T4 = False, nen pipeline dung fp16)')

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
BRIDGE = '/content/drive/MyDrive/father-and-daughter'
LORA_DIR = BRIDGE + '/loras'
!ls -la {BRIDGE}/inbox/

In [ ]:
# 3. Cai dat. ~3 phut. Model tai tu HuggingFace (nhanh hon doc tu Drive nhieu).
!pip install -q diffusers transformers accelerate peft safetensors imageio imageio-ffmpeg hf_transfer
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# 4. Lay code worker. Doi URL repo cua ban vao day.
REPO = 'https://github.com/nguyenkha283/father-daughter-flow.git'
!rm -rf /content/fad && git clone -q father-daughter-flow /content/fad || cp -r {BRIDGE}/code /content/fad
import sys; sys.path.insert(0, '/content/fad')

In [ ]:
# 5. CHAY. Giai doan 1 sinh toan bo keyframe, giai doan 2 sinh clip i2v.
#    Gop ca hai vao 1 phien la co y: tach ra la nhan doi rui ro bi ngat.
from colab.worker import main
main(bridge=BRIDGE, lora_dir=LORA_DIR)

In [ ]:
# 6. Kiem ke ket qua truoc khi dong tab
import os
for root, dirs, files in os.walk(BRIDGE + '/outbox'):
    n = len([f for f in files if f.endswith(('.png','.mp4'))])
    if n: print(f'{root.split("outbox/")[-1]:45s} {n} file')

## Sau khi xong
Tren may 1, chay:
```
rclone sync gdrive:father-and-daughter/outbox ./colab_bridge/outbox
python run.py import jobs/<ten-job>.yaml
python run.py all    jobs/<ten-job>.yaml
```
